# Github Scraping – Blinde Signaturen


In [2]:
import os
import time
import requests
import pandas as pd

from dotenv import load_dotenv
from tqdm import tqdm

import re

load_dotenv()

TOKEN = os.getenv("GITHUB_TOKEN")

HEADERS = {
    "Authorization": f"Bearer {TOKEN}",
    "Accept": "application/vnd.github+json"
}

SEARCH_TERMS = [
    '"blind signature"',
    '"blind signatures"',
    '"anonymous credentials"',
    '"blind"',
    '"blind RSA"',
    '"blind token"',

    '"partially blind signature"',
    '"blind schnorr"',
    '"blind BLS"',                      #Boneh-Lynn-Shacham
    '"VOPRF"',
    '"OPRF" "token"',
    '"RSA blind signatures"',

    '"anonymous tokens"',
    '"private access token"',
    '"trust token"',
    '"private state token"',
    '"unlinkable token"',
    '"challenge bypass"',

    '"blinded message"',
    '"blind issuance"',
]

OUTPUT_FILE = "blind_signature_repos.xlsx"

# Begriffe die zwingend im README stehen müssen, damit ein Repo als relevant gilt
REQUIRED_TERMS = [
    "blind signature",
    "blind signatures",
    "blind signing",
    "blind sign",
    "blind rsa",
    "blind schnorr",
    "blinded message",
    "blinding factor",
    "blind issuance",
    "unblind",
    "partially blind",
    "voprf",
    "anonymous token",
    "private access token",
    "chaumian",
]


def search_repositories(query, per_page=30, max_pages=30):
    url = "https://api.github.com/search/repositories"
    all_items = []

    for page in range(1, max_pages + 1):
        params = {
            "q": f"{query} in:name,description,topics,readme stars:>10",
            "sort": "stars",
            "order": "desc",
            "per_page": per_page,
            "page": page
        }
        response = requests.get(url, headers=HEADERS, params=params)

        if response.status_code == 422:
            break
        if response.status_code != 200:
            print("Error:", response.status_code, response.text)
            break

        items = response.json().get("items", [])
        if not items:
            break

        all_items.extend(items)

    return all_items

def clean_illegal_chars(value):
    """Entfernt Steuerzeichen, die openpyxl nicht in Excel-Zellen erlaubt."""
    if isinstance(value, str):
        # Entfernt ASCII-Steuerzeichen \x00-\x08, \x0b, \x0c, \x0e-\x1f
        return re.sub(r"[\x00-\x08\x0b\x0c\x0e-\x1f]", "", value)
    return value

def get_readme(owner, repo, retries=3):
    url = f"https://api.github.com/repos/{owner}/{repo}/readme"
    headers = HEADERS.copy()
    headers["Accept"] = "application/vnd.github.raw"

    for attempt in range(retries):
        try:
            response = requests.get(url, headers=headers, timeout=15)
            if response.status_code == 200:
                return response.text
            return ""
        except requests.exceptions.RequestException as e:
            print(f"  Timeout/Fehler bei {owner}/{repo} (Versuch {attempt+1}/{retries})")
            time.sleep(5) 

    print(f"  Übersprungen: {owner}/{repo}")
    return ""


def is_relevant(readme_text):
    """Gibt True zurück wenn mindestens ein Pflichtbegriff im README vorkommt."""
    t = readme_text.lower()
    return any(term in t for term in REQUIRED_TERMS)


def classify_readme(text):
    t = text.lower()
    found = [term for term in REQUIRED_TERMS if term in t]
    return {
        "matched_terms":          ", ".join(found),
        "mentions_anti_bot":      any(k in t for k in ["captcha", "bot detection", "challenge bypass", "trust token", "private state token"]),
        "mentions_ecash":         any(k in t for k in ["e-cash", "ecash", "chaumian", "digital cash", "cashu"]),
        "mentions_fraud_prevent": any(k in t for k in ["fraud prevention", "fraud detection", "private reporting", "touchpoint"]),
        "mentions_credentials":   any(k in t for k in ["anonymous credential", "anoncred", "de-identified", "selective disclosure"]),
        "mentions_advertising":   any(k in t for k in ["adveil", "ad measurement", "privacy-preserving ad", "anonymous ad"]),
        "mentions_evoting":       any(k in t for k in ["e-voting", "evoting", "electronic voting", "blind vote"]),
        "mentions_privacy":       "privacy" in t,
        "mentions_authentication": "authentication" in t,
    }


# Duplikat-Tracking über repo_url
seen_urls = set()
all_rows = []
skipped = 0

for term in SEARCH_TERMS:
    repos = search_repositories(term)

    for repo in tqdm(repos, desc=term):
        url = repo["html_url"]

        # Duplikate überspringen
        if url in seen_urls:
            continue
        seen_urls.add(url)
        owner     = repo["owner"]["login"]
        repo_name = repo["name"]
        readme    = get_readme(owner, repo_name)

        # Nicht relevante Repos direkt herausfiltern
        if not is_relevant(readme):
            skipped += 1
            time.sleep(1)
            continue

        analysis = classify_readme(readme)

        row = {
            "search_term": term,
            "repo_name":   repo_name,
            "owner":       owner,
            "stars":       repo["stargazers_count"],
            "forks":       repo["forks_count"],
            "language":    repo["language"],
            "created_at":  repo["created_at"],
            "updated_at":  repo["updated_at"],
            "description": repo["description"],
            "topics":      ", ".join(repo.get("topics", [])),
            "repo_url":    repo["html_url"],
            "readme":      readme[:5000],
        }

        row.update(analysis)
        all_rows.append(row)
        time.sleep(1)

df = pd.DataFrame(all_rows)
df = df.sort_values("stars", ascending=False).reset_index(drop=True)
for col in ["readme", "description", "matched_terms", "topics"]:
    df[col] = df[col].apply(clean_illegal_chars)
df.to_excel(OUTPUT_FILE, index=False)


print(f"Relevante Repos gespeichert: {len(df)}")
print(f"Irrelevante Repos gefiltert: {skipped}")
print(f"Gespeichert in:              {OUTPUT_FILE}")

"anonymous credentials": 100%|██████████| 74/74 [01:38<00:00,  1.33s/it]


Error: 403 {
  "documentation_url": "https://docs.github.com/free-pro-team@latest/rest/overview/rate-limits-for-the-rest-api#about-secondary-rate-limits",
  "message": "You have exceeded a secondary rate limit. Please wait a few minutes before you try again. For more on scraping GitHub and how it may affect your rights, please review our Terms of Service (https://docs.github.com/en/site-policy/github-terms/github-terms-of-service) If you reach out to GitHub Support for help, please include the request ID E004:A530:F8CA7D2:EF4BDCA:6A3B9F2A."
}


"blind schnorr": 100%|██████████| 1/1 [00:00<?, ?it/s]
"blind BLS": 0it [00:00, ?it/s]
"private state token": 100%|██████████| 12/12 [00:13<00:00,  1.15s/it]
"unlinkable token": 0it [00:00, ?it/s]
"blind issuance": 100%|██████████| 12/12 [00:16<00:00,  1.37s/it]


Relevante Repos gespeichert: 238
Irrelevante Repos gefiltert: 1033
Gespeichert in:              blind_signature_repos.xlsx
